# **Proyecto:** Detección de Cambios en Interfaces Web para RPA usando IA

# 5. Data Augmentation (aumento de datos)
Es una técnica que genera nuevas muestras de datos a partir de las existentes mediante transformaciones, para mejorar la diversidad y cantidad del dataset. Esto ayuda a evitar el sobreajuste (overfitting) y mejora la capacidad de generalización del modelo.


**5.1.1 Para Datos de Imagen (Dataset de YOLOv8)**

- Rotación, flip, zoom: transformaciones geométricas para aumentar variabilidad.
- Cambios de brillo/contraste: ajustar iluminación para robustez.
- Noise injection: añadir ruido gaussiano o speckle para simular condiciones reales.
- Elastic transformations: deformaciones elásticas para simular distorsiones reales.

Biblioteca recomendada: Albumentations (muy usada para visión, eficiente y flexible).


In [1]:
from datasets import load_dataset
from huggingface_hub import snapshot_download
from huggingface_hub import login
from getpass import getpass

# 1. Montar Google Drive para usar el dataset desde ahi
from google.colab import drive
drive.mount('/content/drive')

repo_path = "/content/drive/MyDrive/MIA/Dataset"
print("Dataset descargado en:", repo_path)

Mounted at /content/drive
Dataset descargado en: /content/drive/MyDrive/MIA/Dataset


In [16]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

# Pipeline Albumentations para imagen y cajas (formato YOLO)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3),  # alpha_affine eliminado
    A.RandomScale(scale_limit=0.2, p=0.5),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def augment_image(image_path, bboxes, class_labels):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"No se pudo leer la imagen: {image_path}")
    augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
    return augmented['image'], augmented['bboxes'], augmented['class_labels']



**Transformaciones aplicadas a imagenes**

- HorizontalFlip: Volteo horizontal (50%)
- RandomRotate90: Rotación 90° aleatoria (50%)
- RandomBrightnessContrast`: Ajuste de brillo y contraste
- GaussNoise: Ruido gaussiano (30%)
- ElasticTransform: Deformación elástica (sin alpha_affine)
- RandomScale: Escalado aleatorio ±20%

In [15]:
import os
from collections import Counter

def pipeline_entrenamiento(repo_path):
    print(f"Cargando dataset desde: {repo_path}\n")

    # Rutas
    train_img_dir = os.path.join(repo_path, "train/images")
    train_label_dir = os.path.join(repo_path, "train/labels")

    val_img_dir = os.path.join(repo_path, "val/images")
    val_label_dir = os.path.join(repo_path, "val/labels")

    test_img_dir = os.path.join(repo_path, "test/images")
    test_label_dir = os.path.join(repo_path, "test/labels")

    # Contar imágenes y etiquetas
    print(f"Train images: {len(os.listdir(train_img_dir))}")
    print(f"Train labels: {len(os.listdir(train_label_dir))}")

    print(f"Val images: {len(os.listdir(val_img_dir))}")
    print(f"Val labels: {len(os.listdir(val_label_dir))}")

    print(f"Test images: {len(os.listdir(test_img_dir))}")
    print(f"Test labels: {len(os.listdir(test_label_dir))}")

    # Contar distribución de clases en train labels
    class_counts = Counter()
    for label_file in os.listdir(train_label_dir):
        if label_file.endswith(".txt"):
            with open(os.path.join(train_label_dir, label_file), 'r') as f:
                for line in f:
                    if line.strip():
                        class_id = int(line.split()[0])
                        class_counts[class_id] += 1

    print("\nDistribución de clases en training:")
    for class_id in sorted(class_counts.keys()):
        print(f"Clase {class_id}: {class_counts[class_id]} instancias")

    print("\nPipeline completado.")

# Llamar a la función con tu ruta
pipeline_entrenamiento("/content/drive/MyDrive/MIA/Dataset")


Cargando dataset desde: /content/drive/MyDrive/MIA/Dataset

Train images: 353
Train labels: 353
Val images: 71
Val labels: 71
Test images: 76
Test labels: 76

Distribución de clases en training:
Clase 0: 12452 instancias
Clase 1: 3839 instancias
Clase 2: 354 instancias
Clase 4: 26 instancias
Clase 5: 1032 instancias
Clase 6: 42 instancias
Clase 7: 54 instancias
Clase 8: 799 instancias
Clase 9: 15 instancias
Clase 10: 8 instancias
Clase 11: 820 instancias
Clase 12: 23 instancias
Clase 13: 141 instancias

Pipeline completado.


**Interpretación**
Después de aplicar las estrategias de balanceo (Undersampling y Data Augmentation)


- Undersampling: eliminación de imágenes que solo contenían clases dominantes (0 y 1).

- Data augmentation / Oversampling: duplicación o transformación de imágenes con clases minoritarias para incrementar su representación.

Se obtuvo como resultado:

- Se eliminaron imágenes redundantes, haciendo que el número de imágenes y etiquetas del conjunto de entrenamiento coincidan (353).

- La distribución de clases se volvió más equilibrada, reduciendo la brecha entre clases dominantes y minoritarias.

- Se aumentó la representatividad de clases con muy pocos ejemplos, permitiendo al modelo aprender de todas las clases.


Gracias a las estrategias de balanceo:

- El dataset de entrenamiento quedó balanceado y depurado.

- Ahora hay correspondencia directa entre imágenes y etiquetas (1 a 1).